# Archon SigilAGI — live global sentinel and rescue demo

This executable demo is inspired by the Neural-Auto-Patch meme. It fetches live CISA KEV intelligence, performs deterministic evidence scoring, correlates an enrolled asset, and runs a reversible canary rollout. It never probes or changes third-party systems.

In [ ]:
import json, hashlib, time, urllib.request
from dataclasses import dataclass, asdict
from datetime import datetime, timezone

def utc_now():
    return datetime.now(timezone.utc).isoformat()

try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('GPU device:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU fallback')
except Exception:
    torch = None; DEVICE = 'cpu'; print('Torch unavailable; CPU fallback')

CANONICAL_URL='https://raw.githubusercontent.com/BlockChain-BailBonds/archon-sigilagi/main/web/data/alerts.json'
CISA_URL='https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json'
try:
    with urllib.request.urlopen(CANONICAL_URL, timeout=20) as response:
        snapshot=json.loads(response.read(50_000_000))
    feed={'vulnerabilities':[{'cveID':a.get('cve_ids',['unknown'])[0] if a.get('cve_ids') else 'unknown','vendorProject':a.get('vendor','unknown'),'product':a.get('product','unknown')} for a in snapshot.get('alerts',[])]}; feed_mode='CANONICAL ARCHON SIGILAGI SNAPSHOT'; feed_sha256=hashlib.sha256(json.dumps(snapshot,sort_keys=True).encode()).hexdigest()
except Exception:
    try:
        with urllib.request.urlopen(CISA_URL, timeout=20) as response:
            raw=response.read(50_000_000)
        feed=json.loads(raw); feed_mode='LIVE CISA KEV'; feed_sha256=hashlib.sha256(raw).hexdigest()
    except Exception as exc:
        feed={'vulnerabilities':[{'cveID':'CVE-OFFLINE-DEMO','vendorProject':'Example','product':'Example Gateway','vulnerabilityName':'offline captured demonstration'}]}
        feed_mode=f'OFFLINE CAPTURED FALLBACK ({type(exc).__name__})'; feed_sha256='offline-fixture'
print(feed_mode, '| records:', len(feed.get('vulnerabilities', [])), '| sha256:', feed_sha256[:16])


In [ ]:
# Normalize public claims into the control-plane shape.
claims=[]
for item in feed.get('vulnerabilities', []):
    claims.append({
        'cve_id': item.get('cveID','unknown'),
        'vendor': item.get('vendorProject','unknown'),
        'product': item.get('product','unknown'),
        'active_exploitation': True,
        'source': 'CISA KEV',
    })
print('normalized claims:', len(claims))
print('sample:', claims[:2])


In [ ]:
# Enrolled asset inventory. This is the only population eligible for rescue actions.
enrolled_assets=[
    {'asset_id':'tenant-prod-gateway-01','service':'gateway-api','component':'Example Gateway','version':'4.1.6','reachable':True,'telemetry_coverage':.99},
    {'asset_id':'tenant-prod-web-01','service':'web-api','component':'Example Web','version':'2.3.1','reachable':True,'telemetry_coverage':.98},
]

def correlated(claim, asset):
    return claim['product'].lower() in asset['component'].lower() or asset['component'].lower() in claim['product'].lower()

matches=[(c,a) for c in claims for a in enrolled_assets if correlated(c,a)]
print('enrolled assets:', len(enrolled_assets), '| correlated matches:', len(matches))


In [ ]:
# GPU-optimized batch risk feature calculation. The result is deterministic; GPU is acceleration, not authority.
import math
if torch is not None:
    features=torch.tensor([[.96,1.0,1.0 if a['reachable'] else 0.0,a['telemetry_coverage']] for c,a in matches], dtype=torch.float32, device=DEVICE) if matches else torch.empty((0,4),device=DEVICE)
    if len(features):
        weights=torch.tensor([.30,.20,.10,.20], device=DEVICE)
        scores=(features*weights).sum(dim=1).detach().cpu().tolist()
    else: scores=[]
else:
    scores=[.30*.96+.20*1+.10*float(a['reachable'])+.20*a['telemetry_coverage'] for c,a in matches]
print('scores computed:', len(scores), '| device:', DEVICE, '| max:', max(scores) if scores else None)


In [ ]:
AUTONOMOUS={'quarantine_workload','disable_feature_flag','rate_limit_route','block_ioc_temporarily','rollback_container_image'}
def authorize(score, asset, exact_version=False):
    reasons=[]
    if score < .85: reasons.append('score below canary threshold')
    if not exact_version: reasons.append('exact version applicability requires authoritative enrichment')
    if asset['telemetry_coverage'] < .95: reasons.append('telemetry coverage below 95%')
    return {'allow':not reasons,'mode':'canary' if not reasons else 'escalate','reasons':reasons or ['all reversible containment gates passed']}

plans=[]
for (claim,asset),score in zip(matches,scores):
    decision=authorize(score,asset,exact_version=False)
    plans.append({'claim':claim,'asset':asset,'primitive':'quarantine_workload','decision':decision,'created_at':utc_now()})
print('rescue plans:', len(plans)); print(json.dumps(plans[:2], indent=2))


In [ ]:
# Real rollback-safe progressive rollout state machine against a local executor stub.
STAGES=[1,5,20,50,100]
state={'applied':0,'rolled_back':False,'audit':[]}
def apply_stage(percent): state['applied']=percent; state['audit'].append(('stage_started',percent))
def evaluate(percent): return (percent <= 20, 'healthy baseline' if percent <= 20 else 'simulated abort threshold')
def rollback(): state['rolled_back']=True; state['applied']=0; state['audit'].append(('rollback',0))
for percent in STAGES:
    apply_stage(percent); healthy,reason=evaluate(percent); state['audit'].append(('evaluated',percent,healthy,reason))
    if not healthy: rollback(); break
print(json.dumps(state, indent=2))
assert state['rolled_back'] and state['applied']==0


## Ecosystem watchlist and rescue reporting

The global sentinel watches public advisories for major software ecosystems, including Hugging Face, X, GitHub, Cloudflare, Kubernetes, Docker, AWS, Microsoft, Google, OpenAI, Linux, Nginx, Apache, Python, npm, and PyPI. “Watch” means public threat-intelligence correlation only; it does not scan, log into, exploit, or modify those companies or sites.

In [ ]:
WATCHLIST={
    'Hugging Face':['hugging face','transformers','safetensors','text-generation-inference'],
    'X':['twitter','x.com'],
    'GitHub':['github','actions','ghes'],
    'Cloudflare':['cloudflare'],
    'Kubernetes':['kubernetes','k8s'],
    'Docker':['docker','containerd'],
    'AWS':['amazon web services','aws','eks'],
    'Microsoft':['microsoft','windows','azure','exchange'],
    'Google':['google','chrome','gcp'],
    'OpenAI':['openai'],
    'Linux':['linux','kernel'],
    'Web stack':['nginx','apache','node.js','python','php','openssl'],
}

def ecosystem_matches(claim):
    text=(claim['vendor']+' '+claim['product']).lower()
    return [name for name,terms in WATCHLIST.items() if any(term in text for term in terms)]

watch_reports=[]
for claim in claims:
    ecosystems=ecosystem_matches(claim)
    if ecosystems:
        watch_reports.append({'cve_id':claim['cve_id'],'ecosystems':ecosystems,'vendor':claim['vendor'],'product':claim['product'],'source':claim['source'],'active_exploitation':claim['active_exploitation']})
print('watchlist threat reports:',len(watch_reports))
for report in watch_reports[:20]: print(report)


In [ ]:
# Rescue-plan report: plans are generated only for explicitly enrolled assets.
rescue_reports=[]
for report in watch_reports:
    for asset in enrolled_assets:
        if any(term in asset['component'].lower() for ecosystem in report['ecosystems'] for term in WATCHLIST[ecosystem]):
            rescue_reports.append({
                'asset_id':asset['asset_id'], 'service':asset['service'], 'cve_id':report['cve_id'],
                'recommended_primitive':'quarantine_workload',
                'scope':'1% canary', 'rollback':'automatic',
                'status':'escalate_for_exact_version_and_telemetry'
            })
print('enrolled-asset rescue reports:',len(rescue_reports))
print(json.dumps(rescue_reports[:20],indent=2))


## Result

The notebook has demonstrated the complete safe decision path: public intelligence → normalized claim → enrolled-asset correlation → GPU-accelerated deterministic scoring → constrained rescue plan → canary health gate → automatic rollback. A production executor is intentionally not invoked from Kaggle; production actions require the enrolled tenant’s signed connector and authorization policy.